# Valuation-to-Option Pricing Lab

This notebook keeps two valuation tracks separate:

1. **Analyst-scenario discounted expected payoff:** five terminal equity-value scenarios with subjective weights.
2. **CRR risk-neutral lattice value:** a no-arbitrage option-pricing model built through repeated two-state replication.

The five analyst scenarios are not an expanded binomial tree and are not risk-neutral probabilities. They answer a different question: *Given these valuation beliefs and weights, what is the discounted expected option payoff?*

## 1. Load and validate student inputs

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
if not 0<=M['risk_free_rate']<=1: raise ValueError('Enter risk_free_rate in decimal form, such as 0.045.')
if not 0<M['volatility']<=3: raise ValueError('Enter volatility in decimal form, such as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one horizon.')
M

{'company': 'Apple Inc.', 'ticker': 'AAPL', 'valuation_date': '2026-11-11', 'spot': 355.0, 'strike': 355.0, 'maturity_years': 0.4273972602739726, 'risk_free_rate': 0.0438, 'dividend_yield': 0.0030422535211267, 'volatility': 0.257, 'volatility_type': 'implied', 'option_type': 'call', 'exercise_style': 'american', 'lattice_steps': 200, 'market_bid': 26.7, 'market_ask': 27.3, 'dcf_value': 97.2, 'comps_value': 194.6, 'expiration': '2027-04-16', 'contract_multiplier': 100, 'quote_time_et': '14:30', 'volume': 35, 'open_interest': 1409, 'market_mid': 27.0}

## 2. Equity valuation and analyst distribution

In [2]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
pd.Series(valuation)

Current market          355.0
Comparable companies    194.6
Student DCF              97.2
Analyst median          310.0
dtype: float64

## 3. Five analyst-informed valuation states

The state prices are reproducible quantiles of the analyst-target distribution. The weights are disclosed subjective scenario masses, not risk-neutral probabilities.

- Severe bear: 5th percentile, 10% weight
- Bear: 25th percentile, 20% weight
- Base: 50th percentile, 40% weight
- Bull: 75th percentile, 20% weight
- Tail: 95th percentile, 10% weight

In [3]:
quantile_map={'Severe bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}
weights={'Severe bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({
    'state':list(quantile_map),
    'quantile':list(quantile_map.values()),
    'terminal_price':[float(targets.target.quantile(q)) for q in quantile_map.values()],
    'subjective_weight':[weights[k] for k in quantile_map]
})
assert np.isclose(states.subjective_weight.sum(),1.0)
states

,state,quantile,terminal_price,subjective_weight
0,Severe bear,0.05,260.0,0.1
1,Bear,0.25,300.0,0.2
2,Base,0.50,310.0,0.4
3,Bull,0.75,310.0,0.2
4,Tail,0.95,366.0,0.1


## 4. Analyst-scenario payoff decomposition

In [4]:
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float)
    return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type'])
states['weighted_payoff']=states.subjective_weight*states.option_payoff
weighted_terminal_payoff=float(states.weighted_payoff.sum())
discount_factor=float(np.exp(-M['risk_free_rate']*M['maturity_years']))
belief_value=discount_factor*weighted_terminal_payoff
weight_positive=float(states.loc[states.option_payoff>0,'subjective_weight'].sum())
weight_zero=1-weight_positive
print(f'Weight producing zero payoff: {weight_zero:.1%}')
print(f'Weight producing positive payoff: {weight_positive:.1%}')
print(f'Weighted terminal payoff: ${weighted_terminal_payoff:,.2f}')
print(f'Discount factor: {discount_factor:.4f}')
print(f'Analyst-scenario discounted expected payoff: ${belief_value:,.2f}')
states

Weight producing zero payoff: 90.0%
Weight producing positive payoff: 10.0%
Weighted terminal payoff: $1.10
Discount factor: 0.9815
Analyst-scenario discounted expected payoff: $1.08


,state,quantile,terminal_price,subjective_weight,option_payoff,weighted_payoff
0,Severe bear,0.05,260.0,0.1,0.0,0.0
1,Bear,0.25,300.0,0.2,0.0,0.0
2,Base,0.50,310.0,0.4,0.0,0.0
3,Bull,0.75,310.0,0.2,0.0,0.0
4,Tail,0.95,366.0,0.1,11.0,1.1


### Interpretation
A low analyst-scenario value is intuitive when much of the assigned weight ends below the call strike or above the put strike. The payoff contribution is visible state by state. This is not a replication price.

## 5. CRR risk-neutral lattice

In [5]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
lattice

{'price': 26.689058782213493, 'u': 1.0119513333503725, 'd': 0.9881898141180327, 'p_star': 0.5006956122940549, 'dt': 0.002136986301369863}

## 6. Theoretical validation and distribution comparison

In [6]:
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]
conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
analyst_log_dispersion=float(np.log(targets.target/M['spot']).std(ddof=1))
vol_horizon_dispersion=float(M['volatility']*np.sqrt(M['maturity_years']))
distribution_message='Analyst targets are narrower than the volatility input used by CRR.' if analyst_log_dispersion<vol_horizon_dispersion else 'Analyst targets are at least as dispersed as the volatility input used by CRR.'
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
print(distribution_message)
validation

Analyst targets are narrower than the volatility input used by CRR.


{'valid_risk_neutral_probability': True, 'inside_theoretical_bounds': True, 'put_call_parity_pass': True, 'convergence_pass': True, 'parity_residual': 6.277645070440485e-12, 'convergence_gap': 0.01772576468738407}

## 7. Export dashboard data

In [7]:
results={
 'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,
 'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,
 'belief_decomposition':{'weight_zero_payoff':weight_zero,'weight_positive_payoff':weight_positive,'weighted_terminal_payoff':weighted_terminal_payoff,'discount_factor':discount_factor},
 'distribution_comparison':{'analyst_log_dispersion':analyst_log_dispersion,'crr_horizon_volatility':vol_horizon_dispersion,'message':distribution_message},
 'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],
 'comparison':{'Analyst-scenario discounted expected payoff':belief_value,'CRR risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']}
}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2))
print('Exported docs/model_results.json')
results['comparison']

Exported docs/model_results.json


{'Analyst-scenario discounted expected payoff': 1.0795995440230606, 'CRR risk-neutral lattice value': 26.689058782213493, 'Observed market midpoint': 27.0}

## 8. Decision memo prompt

Explain why the analyst-scenario payoff and CRR value differ. Address the share of scenario weight producing a zero payoff, the contribution from positive-payoff states, the strike, analyst-target dispersion, the CRR volatility input, and why neither estimate should automatically be characterized as the uniquely correct market price.

## 9. Module 5 Greeks, validation and full repricing

The primary Greeks use a European Black–Scholes–Merton benchmark at the same inputs and valuation date as CRR. CRR prices the listed American contract. The two full repricings separate Taylor approximation error from lattice and exercise differences. Greek values apply to one long option share.

In [8]:
from math import erf, exp, log, pi, sqrt
from IPython.display import display

S, K, T, r, v, q = (M[k] for k in ('spot','strike','maturity_years','risk_free_rate','volatility','dividend_yield'))
kind = M['option_type']
american = M['exercise_style'] == 'american'
multiplier = int(M['contract_multiplier'])
N = M['lattice_steps']
sign = 1 if kind == 'call' else -1

def benchmark(s, t, vol, rate):
    d1 = (log(s/K)+(rate-q+vol*vol/2)*t)/(vol*sqrt(t))
    d2 = d1-vol*sqrt(t)
    normal = lambda x: (1+erf(x/sqrt(2)))/2
    density = exp(-d1*d1/2)/sqrt(2*pi)
    sq, kr = s*exp(-q*t), K*exp(-rate*t)
    first, second = normal(sign*d1), normal(sign*d2)
    return {
        'price': sign*(sq*first-kr*second),
        'delta': sign*exp(-q*t)*first,
        'gamma': exp(-q*t)*density/(s*vol*sqrt(t)),
        'vega': sq*density*sqrt(t)/100,
        'theta': (-sq*density*vol/(2*sqrt(t))+sign*(q*sq*first-rate*kr*second))/365,
        'rho': sign*kr*t*second/100
    }

def euro(s=S,t=T,vol=v,rate=r):
    return benchmark(s,t,vol,rate)['price']

base = benchmark(S,T,v,r)
def differences(s_bump, v_bump, day_bump, r_bump):
    elapsed = day_bump/365
    up, down = euro(s=S+s_bump), euro(s=S-s_bump)
    return [
        (up-down)/(2*s_bump), (up-2*base['price']+down)/s_bump**2,
        (euro(vol=v+v_bump)-euro(vol=v-v_bump))/(200*v_bump),
        (euro(t=T-elapsed)-euro(t=T+elapsed))/(2*day_bump),
        (euro(rate=r+r_bump)-euro(rate=r-r_bump))/(200*r_bump)
    ]
names = ['delta','gamma','vega','theta','rho']
units = ['shares per option share','delta change per $1 spot','USD per volatility percentage point','USD per calendar day elapsed','USD per rate percentage point']
greeks = pd.DataFrame({'Greek':names,'Units':units,'Analytic':[base[k] for k in names],
                       'Bump':['$0.10','$0.10','0.10 percentage point','1 calendar day','0.10 percentage point'],
                       'Central difference':differences(.1,.001,1,.001),
                       'Half-bump check':differences(.05,.0005,.5,.0005)})
greeks['Difference'] = greeks.Analytic-greeks['Central difference']
greeks['Per contract'] = greeks.Analytic*multiplier
assert (0 <= base['delta'] <= 1 if kind == 'call' else -1 <= base['delta'] <= 0)
assert base['gamma'] >= 0
assert np.allclose(greeks.Analytic,greeks['Central difference'],atol=1e-5,rtol=1e-3)
display(greeks.round(8))

def reprice_crr(s,t,vol,rate,steps=N,early=american):
    return crr(s,K,t,rate,vol,steps,q,kind,early)['price']

american_convergence = pd.DataFrame({'Steps':steps,'European CRR':conv,
    'Selected style CRR':[reprice_crr(S,T,v,r,n) for n in steps]})
display(american_convergence)
european_crr = reprice_crr(S,T,v,r,early=False)
early_premium = lattice['price']-european_crr
if american:
    assert early_premium >= -1e-8
    assert max(0,sign*(S-K)) <= lattice['price'] <= (S if kind=='call' else K)
horizon_months = int(targets.horizon_months.iloc[0])
horizon_gap = 12*T-horizon_months
print(f'Option maturity {12*T:.2f} months versus analyst horizon {horizon_months} months. Difference {horizon_gap:.2f} months.')
print(f'Early exercise premium under continuous yield: ${early_premium:.6f} per share.')
print(f'European Black–Scholes benchmark: ${base["price"]:.4f} per share.')

def test_move(label,ds,dvol,days,dr):
    delta = base['delta']*ds
    curvature = .5*base['gamma']*ds**2
    total = delta+curvature+base['vega']*100*dvol+base['theta']*days+base['rho']*100*dr
    exact = euro(S+ds,T-days/365,v+dvol,r+dr)-base['price']
    crr_move = reprice_crr(S+ds,T-days/365,v+dvol,r+dr)-lattice['price']
    return [label,ds,100*dvol,days,100*dr,delta,delta+curvature,total,exact,crr_move,
            delta-exact,delta+curvature-exact,total-exact]
repricing = pd.DataFrame([
    test_move('Spot only',10,0,0,0),
    test_move('Volatility only',0,.01,0,0),
    test_move('One day elapsed',0,0,1,0),
    test_move('Rate only',0,0,0,.0025),
    test_move('Combined',5,.01,1,.0025)],
    columns=['Scenario','Spot bump ($)','IV bump (pp)','Days elapsed','Rate bump (pp)','Delta only','Delta-gamma',
             'All Greeks','Full BSM change','Full CRR change','Delta error','Delta-gamma error','All Greeks error'])
display(repricing.round(6))
grid = np.unique(np.r_[np.linspace(min(S,K)*.70,max(S,K)*1.30,101),S,K])
curves = [{'spot':float(s),**{k:benchmark(float(s),T,v,r)[k] for k in names}} for s in grid]
results.update(greeks=greeks.to_dict('records'),greek_curves=curves,repricing=repricing.to_dict('records'),
               european_bsm=base['price'],early_exercise_premium=early_premium,
               selected_style_convergence=american_convergence.to_dict('records'),
               analyst_horizon_months=horizon_months,horizon_gap_months=horizon_gap)
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2,allow_nan=False))

Option maturity 5.13 months versus analyst horizon 12 months. Difference -6.87 months.
Early exercise premium under continuous yield: $0.000000 per share.
European Black–Scholes benchmark: $26.7186 per share.


,Greek,Units,Analytic,Bump,Central difference,Half-bump check,Difference,Per contract
0,delta,shares per option share,0.573693,$0.10,0.573693,0.573693,7.000000e-08,57.369263
1,gamma,delta change per $1 spot,0.006563,$0.10,0.006563,0.006563,0.000000e+00,0.656325
2,vega,USD per volatility percentage point,0.908532,0.10 percentage point,0.908532,0.908532,9.000000e-08,90.853207
3,theta,USD per calendar day elapsed,-0.094373,1 calendar day,-0.094373,-0.094373,3.900000e-07,-9.437299
4,rho,USD per rate percentage point,0.756246,0.10 percentage point,0.756246,0.756246,2.000000e-07,75.624644


,Steps,European CRR,Selected style CRR
0,25,26.950104,26.950104
1,50,26.600636,26.600636
2,100,26.659546,26.659546
3,200,26.689059,26.689059
4,500,26.706785,26.706785


,Scenario,Spot bump ($),IV bump (pp),Days elapsed,Rate bump (pp),Delta only,Delta-gamma,All Greeks,Full BSM change,Full CRR change,Delta error,Delta-gamma error,All Greeks error
0,Spot only,10,0.0,0,0.00,5.736926,6.065089,6.065089,6.057977,6.090513,-0.321051,0.007111,0.007111
1,Volatility only,0,1.0,0,0.00,0.000000,0.000000,0.908532,0.908589,0.907454,-0.908589,-0.908589,-0.000057
2,One day elapsed,0,0.0,1,0.00,0.000000,0.000000,-0.094373,-0.094499,-0.094405,0.094499,0.094499,0.000126
3,Rate only,0,0.0,0,0.25,0.000000,0.000000,0.189062,0.189432,0.189429,-0.189432,-0.189432,-0.000371
4,Combined,5,1.0,1,0.25,2.868463,2.950504,3.953724,3.954825,4.013086,-1.086361,-1.004321,-0.001100


29998

## Integrated decision memo

### Equity Valuation

The November 11 snapshot placed Apple at $355.00. My prior DCF was $97.20 per share, far below that price. That estimate came from an earlier class submission, so I would verify its original valuation date and diluted share count before relying on the size of the gap. Terminal growth mattered most in the original DCF sensitivity work, and the constant operating margin also limits how confidently I can carry the estimate forward. The broad technology peer estimate is $194.60, based on the median trailing P/E of Microsoft, Alphabet, and Meta multiplied by Apple’s trailing earnings per share. These firms offer a technology-sector reference, but Apple’s hardware mix, services exposure, and capital returns differ from theirs. The five analyst targets average $310 and have a median of $310. Their range is $250 to $380, with three observations between $300 and $310. The targets were reported November 3–6 and may share earnings assumptions. The twelve-month horizon is assumed from the provider’s convention and needs confirmation from the individual reports.

### Option Valuation

I selected the April 16, 2027 American $355 Apple call. The same intraday snapshot showed a $26.70 bid, $27.30 ask, and $27.00 midpoint per share. The 200-step CRR value is $26.6891, about $0.31 below the midpoint and roughly one cent below the bid. The snapshot showed 35 contracts of volume and 1,409 open interest, so the quote is useful evidence, although available size and the subsequent quote can change. The discounted analyst-scenario payoff is about $1.08. Ninety percent of the scenario weight produces zero payoff, and the upper-tail state supplies all of the weighted payoff. The lattice uses volatility and a risk-neutral probability determined by the carrying cost. The analyst calculation uses subjective weights and a small sample of price opinions. The scenario horizon is also about seven months longer than the option maturity. Together these differences make the scenario value a weak benchmark for the quoted premium.

### Greeks and Risk Profile

At the stated inputs, the European benchmark delta is 0.5737 and gamma is 0.00656 per dollar of spot movement. A $1 increase in Apple stock initially adds about $0.57 per option share, or $57.37 per standard contract. The gamma means that the call gains stock exposure as Apple rises. Curvature is strongest around the near-the-money region and generally becomes more concentrated as expiration approaches. Vega is $0.9085 per volatility percentage point, or about $90.85 per contract. Daily theta is minus $0.0944 per share, and rho is positive $0.7562 per rate percentage point. For the tested $10 stock move, delta dominates the dollar response. For the market comparison, volatility is the assumption I would monitor most closely because even a modest IV change can exceed the CRR-to-midpoint gap. A first-order hedge for one long contract would short roughly 57 shares and would require rebalancing as delta changes.

### Approximation and Model Risk

For a $10 increase in Apple, delta alone predicts $5.7369 per share and delta-gamma predicts $6.0651. Full European repricing gives $6.0580, while the American lattice changes by $6.0905. For a $5 stock rise, one volatility point increase, one day elapsed, and a quarter-point rate rise, the all-Greek estimate is $3.9537 versus $3.9548 from full European repricing and $4.0131 from the lattice. Adding curvature improves the stock-only estimate, and incorporating volatility, time, and rates improves the combined estimate. The small errors are specific to these bumps. Larger moves change the Greeks and make cross-effects more important. Continuous dividends simplify Apple’s discrete payout schedule. The model also assumes constant volatility and a rate proxy, while real quotes reflect liquidity, skew, and how the market values the early-exercise feature. The American and European values are nearly identical under these inputs, although discrete dividend timing could matter for a deeper in-the-money call.

### Conclusion

The selected call is close to the American lattice benchmark, with a midpoint about $0.31 higher per share. The largest uncertainty in that comparison is the volatility input and its connection to the observed quote. I would revise the conclusion toward a material model-to-market gap if a fresh same-contract midpoint stayed at least $1.00 away from a rerun lattice value using contemporaneous spot and IV. A gap of that size would warrant investigating spread, dividend timing, and model assumptions before interpreting it as a pricing difference.

### AI-use disclosure draft for student review

I used ChatGPT to organize sources, extend the existing notebook, develop visualizations, and check calculations. The notebook shows the original assertions, central-difference checks, half-bump comparisons, and full repricings. Before submitting, I will verify the contract snapshot, original DCF assumptions and valuation date, and each target horizon, and revise the interpretation to reflect my own analysis. Update this disclosure with the verification and edits you actually complete.
